# Driving-event mining on KITTI — Colab walkthrough

This notebook runs the full pipeline on the KITTI tracking benchmark:

1. vehicle-dynamics signals (OXTS) → quality checks → filtering → rule-based events
2. camera frames → YOLOv8 → tracking → lead vehicle → time-to-collision (TTC), checked against the 3D labels
3. sliding-window features → anomaly detection → review candidates
4. manual review of the candidates → rule precision and a classifier trained on reviewed labels

**Runtime:** `Runtime → Change runtime type → T4 GPU` speeds up YOLO. Everything else runs on CPU.

## 1 · Setup

In [ ]:
# Replace with your GitHub user name once the repo is pushed
REPO = "https://github.com/sandraesudhakaran-creator/kitti-drive-event-mining.git"
!git clone -q $REPO /content/kitti-drive-event-mining
%cd /content/kitti-drive-event-mining
!pip install -q -e ".[video,review]"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Results and review labels are written to Drive, so they survive a disconnect.
WORK = "/content/drive/MyDrive/drive_events"
!mkdir -p $WORK

## 2 · Sanity check on synthetic data (1 minute)

Synthetic drives in exact KITTI format with known ground truth. If this recovers the injected events, the installation works.

In [ ]:
!drive-events synth --out /content/synthetic --n 3 --seed 1
!drive-events run --data-root /content/synthetic --out /content/out_synthetic --detector ground_truth --max-cards 5
!drive-events evaluate --events /content/out_synthetic/events.csv --truth /content/synthetic/truth_events.csv

## 3 · Get the KITTI tracking data

1. Register at <https://www.cvlibs.net/datasets/kitti/user_register.php> (free, non-commercial licence) and log in.
2. Open <https://www.cvlibs.net/datasets/kitti/eval_tracking.php>. For each of the four downloads below, **right-click the link → Copy link address** and paste it into the cell:
   - *Download left color images of tracking data set* → `data_tracking_image_2.zip` (≈15 GB)
   - *Download GPS/IMU data* → `data_tracking_oxts.zip`
   - *Download camera calibration matrices* → `data_tracking_calib.zip`
   - *Download training labels* → `data_tracking_label_2.zip`
3. The zips go to the Colab disk (not Drive — the image zip is about as big as a free Drive). Colab forgets this disk when the session ends, so this cell is re-run in each new session; it takes a few minutes. Results and labels are saved on Drive and are kept.

In [ ]:
LINKS = {
    "data_tracking_oxts":    "PASTE-LINK-HERE",
    "data_tracking_label_2": "PASTE-LINK-HERE",
    "data_tracking_calib":   "PASTE-LINK-HERE",
    "data_tracking_image_2": "PASTE-LINK-HERE",
}
ZIPS, DATA = "/content/kitti_zips", "/content/kitti_tracking"
!mkdir -p $ZIPS $DATA
for name, url in LINKS.items():
    assert url.startswith("http"), f"paste the download link for {name}"
    !wget -q -nc -O {ZIPS}/{name}.zip "{url}"
    # only the training split has labels, so testing/ is skipped
    !unzip -q -o {ZIPS}/{name}.zip "training/*" -d $DATA
!ls $DATA/training
!ls $DATA/training/oxts | wc -l   # expect 21

## 4 · Run the pipeline

Two runs, so the effect of the detector can be separated from the rest of the pipeline:

* **`yolo`** — the realistic setting, a pretrained YOLOv8 finds the vehicles
* **`ground_truth`** — the labelled boxes act as a perfect detector (upper bound)

YOLO detections are cached in the output folder, so re-runs with other thresholds are fast.

In [ ]:
!drive-events run --data-root $DATA --out $WORK/run_yolo --detector yolo

In [ ]:
!drive-events run --data-root $DATA --out $WORK/run_oracle --detector ground_truth --no-cards

KITTI drives are calm city driving, so fixed comfort thresholds may fire rarely. The percentile mode sets thresholds from the data distribution instead (never softer than the fixed exit levels). Compare both and decide which suits your analysis — and explain why in the README.

In [ ]:
!drive-events run --data-root $DATA --out $WORK/run_percentile --detector yolo --threshold-mode percentile --no-cards

## 5 · Look at the results

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

RUN = f"{WORK}/run_yolo"
display(Markdown(open(f"{RUN}/summary.md").read()))

In [ ]:
display(Image(f"{RUN}/figures/overview.png"))
display(Image(f"{RUN}/figures/ttc_scatter.png"))

In [ ]:
# Timeline of one drive: signals on top, detected events below
display(Image(f"{RUN}/figures/timeline_0000.png"))

In [ ]:
events = pd.read_csv(f"{RUN}/events.csv", dtype={"sequence": str})
events.groupby(["type", "severity"]).size().unstack(fill_value=0)

In [ ]:
# YOLO vs. perfect detector: how much does detection quality cost the TTC estimate?
for name in ["run_yolo", "run_oracle"]:
    s = open(f"{WORK}/{name}/summary.md").read()
    print(f"== {name}"); print(s[s.find("## Camera TTC"):s.find("## Thresholds")])

In [ ]:
# Data-quality findings (gaps, faulty samples, IMU vs. speed consistency)
pd.read_csv(f"{RUN}/quality.csv", dtype={"sequence": str})

## 6 · Review the event candidates

Each card shows the camera at start / peak / end and the signals around the event.
Decide for each: **Accept** (rule was right), **Reject** (false alarm) or **Relabel** (real event, wrong type).
Every click is saved immediately to `labels.csv` on your Drive.

Aim for at least ~100 reviewed events across several sequences; the cards cover the first `max_cards` events (default 60) — raise it with `--max-cards` if you want more.

In [ ]:
from drive_events.labeling import LabelStore, review_events

store = LabelStore(f"{WORK}/labels.csv")
review_events(events, store, cards_dir=f"{RUN}/cards", reviewer="sandra")

In [ ]:
# Also look at what the anomaly detector found that the rules missed
cands = pd.read_csv(f"{RUN}/review_candidates.csv", dtype={"sequence": str})
for eid in cands["event_id"].head(5):
    display(Image(f"{RUN}/cards/candidates/{eid}.png"))

## 7 · Evaluate with the reviewed labels

* **Rule precision** = share of rule events the reviewer accepted, per type.
* **Window classifier** trained on reviewed labels, cross-validated **by sequence** (no leakage between overlapping windows of the same drive), compared with a majority-class baseline.

In [ ]:
!drive-events evaluate-labels --run $RUN --labels $WORK/labels.csv

## 8 · Numbers for the README

Copy the real values from the outputs above into the *Results* section of the README:
rule precision per type, TTC median error and critical-warning precision/recall (YOLO vs. oracle), classifier macro-F1 vs. baseline, and one or two example cards and the timeline figure.